In [1]:
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk


reader = GithubRepositoryDataReader(
    repo_owner="spring-projects",
    repo_name="spring-ai",
    commit_id="main",             # or a specific commit SHA / tag like your example
    allowed_extensions={"adoc"},   # Spring AI docs use .adoc, not .md
    filename_filter=lambda
    path: "/pages/" in path,
)
files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

chunks = chunk_documents(documents, size=2000, step=1000)

# Initialize Elasticsearch client
es = Elasticsearch("http://localhost:9200")

INDEX_NAME = "spring-ai-docs"

if not es.indices.exists(index=INDEX_NAME):
    es.indices.create(
        index=INDEX_NAME,
        mappings={
            "properties": {
                "content": {"type": "text"},
                "filename": {"type": "keyword"},
                "start": {"type": "integer"},
            }
        },
    )

print("Index ready.")

# Ingest chunks into Elasticsearch

actions = [
    {
        "_index": INDEX_NAME,
        "_id": i,
        "_source": {
            "content": chunk["content"],
            "filename": chunk["filename"],
            "start": chunk["start"],
        },
    }
    for i, chunk in enumerate(chunks)
]

bulk(es, actions)

print(f"Indexed {len(actions)} chunks.")




Index ready.
Indexed 1633 chunks.


In [2]:
print(es.cat.indices(format="json"))

[{'health': 'yellow', 'status': 'open', 'index': 'spring-ai-docs', 'uuid': 'llQ0IRcFQ_u5-a2DoNa_EQ', 'pri': '1', 'rep': '1', 'docs.count': '1633', 'docs.deleted': '0', 'store.size': '1.8mb', 'pri.store.size': '1.8mb', 'dataset.size': '1.8mb'}]


In [3]:
response = es.search(
    index="spring-ai-docs",
    query={"match_all": {}},
    size=3,
)

for hit in response["hits"]["hits"]:
    print(hit["_source"])
    print("-" * 80)

{'content': '[[Advisors-Recursive]]\n\n= Recursive Advisors\n\n== What is a Recursive Advisor?\n\nimage:advisors-recursive.png[Advisors Recursive, width=230, float="right", align="center", alt="Advisors Recursive"]\nRecursive advisors are a special type of advisor that can loop through the downstream advisor chain multiple times.\nThis pattern is useful when you need to repeatedly call the LLM until a certain condition is met, such as:\n\n* Executing tool calls in a loop until no more tools need to be called\n* Validating structured output and retrying if validation fails\n* Implementing evaluation logic with modifications to the request\n* Implementing retry logic with modifications to the request\n\nThe `CallAdvisorChain.copy(CallAdvisor after)` method is the key utility that enables recursive advisor patterns.\nIt creates a new advisor chain that contains only the advisors that come after the specified advisor in the original chain\nand allows the recursive advisor to call this sub-